   # Epoch selection

   Selects the number of training epochs for the centralized baseline. Each model is
   trained from the same seed and scored on the **pooled validation sets of all five
   hospitals**. The global test set is not used here. Chosen value: **8 epochs**
   (see `configs/config.yaml`).

In [ ]:
import sys; sys.path.append("..")
import numpy as np, torch
from src.model import build_model, train, predict_proba
from src.centralized import load_combined_train
from src.data_loader import load_config, get_input_dim, load_hospital_data
from src.metrics import compute_metrics

cfg = load_config(); t = cfg["training"]
X, y = load_combined_train(5)
val = [load_hospital_data(h)[2:] for h in range(5)]
Xv = np.concatenate([v[0] for v in val]); yv = np.concatenate([v[1] for v in val])

for ep in [3, 5, 8, 10, 15, 20, 30]:
    torch.manual_seed(cfg["seed"])
    m = build_model(get_input_dim(), cfg)
    train(m, X, y, epochs=ep, lr=t["learning_rate"], batch_size=t["batch_size"],
          cfg=cfg, seed=cfg["seed"])
    r = compute_metrics(yv, predict_proba(m, Xv))
    print(ep, round(r["roc_auc"], 4))

3 0.6657
5 0.6699
8 0.672
10 0.6646
15 0.6592
20 0.6492
30 0.6348
